# Game Simulator
Forecast DAU, payer DAU, and revenue over a 365-day horizon by adjusting UA spend, CPI, retention, and conversion inputs.

In [1]:
# show
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from common_lib.sql import BigQueryConnector
from common_lib.sheets import load_inputs, get_inputs_dir
from common_lib.simulation import (
    SimulationEngine, PlatformInputs,
    save_scenario, load_scenario, list_scenarios,
    save_result, load_result, list_results,
)
from common_lib.widgets import ScenarioPanel
from common_lib.tables import monthly_table, comparison_table, export_all_tables, comparison_table

#print('Inputs dir:', get_inputs_dir())

In [ ]:
refresh_data = False  # Set to True to refresh data from BigQuery, False to load from local pickle

In [3]:
# show
actuals_params = {'start_date': '2021-06-01'}
bqc = BigQueryConnector()
#cost = bqc.print_cost_estimate('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)


In [4]:
PLATFORM_MAP = {'AND': 'android', 'IOS': 'ios'}

if refresh_data == True:
    actuals = bqc.get('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)
    pd.to_pickle(actuals, './data/actuals.pkl')
else:
    actuals = pd.read_pickle('./data/actuals.pkl')

actuals['dt'] = pd.to_datetime(actuals['dt'])
actuals['platform'] = actuals['platform'].map(PLATFORM_MAP).fillna(actuals['platform'].str.lower())
actuals = actuals.sort_values('dt')

# Anchor DAU: last observed day per platform
anchor_dau = actuals.sort_values('dt').groupby('platform')['dau'].last().to_dict()

In [5]:
# show
cohort_params = {'start_date': '2021-06-01'}
bqc = BigQueryConnector()
#cost = bqc.print_cost_estimate('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
#cost = bqc.print_cost_estimate('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)

In [6]:


if refresh_data == True:
    live_retention  = bqc.get('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
    live_retention.to_pickle('./data/live_retention.pkl')
    live_conversion = bqc.get('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)
    live_conversion.to_pickle('./data/live_conversion.pkl')
else:
    live_retention  = pd.read_pickle('./data/live_retention.pkl')
    live_conversion = pd.read_pickle('./data/live_conversion.pkl')

live_retention['platform']  = live_retention['platform'].map(PLATFORM_MAP).fillna(live_retention['platform'].str.lower())
live_conversion['platform'] = live_conversion['platform'].map(PLATFORM_MAP).fillna(live_conversion['platform'].str.lower())

In [7]:
# show
sheet_inputs = load_inputs()

#for name, df in sheet_inputs.items():
    #print(f'\n--- {name} ---')
    #print(df.to_string(index=False))

In [8]:
# show
from common_lib.app import prefill_panel, setup_callbacks

engine = SimulationEngine()
panel  = ScenarioPanel(saved_scenarios=list_scenarios())
prefill_panel(panel, actuals, anchor_dau, sheet_inputs)
setup_callbacks(panel, engine, actuals,
                live_retention=live_retention, live_conversion=live_conversion,
                installs=actuals)
panel.display()

<IPython.core.display.Javascript object>

In [9]:
from common_lib.plots import plot, plot_retention, plot_conversion, configure as configure_plots
from common_lib.simulation import list_results

configure_plots(actuals)

# ── Usage ──────────────────────────────────────────────────────────────────
# plot('base_case')                         # all charts
# plot('base_case', chart='dau')            # DAU only
# plot('base_case', chart='revenue')        # daily revenue
# plot('base_case', chart='monthly')        # monthly bar
# plot(['base_case', 'high_ua'])            # compare scenarios
#
# plot_retention('base_case')               # retention curve from saved scenario
# plot_conversion('base_case')              # conversion curve from saved scenario
# plot_retention(['base_case', 'high_ua'])  # compare retention curves across scenarios
# plot_retention(panel.get_curve_anchors()) # preview current panel state
#print('Available results:', list_results())

In [10]:
#plot('test3', chart='dau')

In [11]:
#plot('test3', chart='revenue')

In [ ]:


results = comparison_table(actuals=actuals)

#round results for better display on pandas
results['Cumul Margin 2027-03'] = results['Cumul Margin 2027-03'].round(0)
results['Cumul Margin 2027-12'] = results['Cumul Margin 2027-12'].round(0)


In [16]:
results.sort_values('Cumul Margin 2027-12', ascending=False)

,DAU 2027-03,Cumul Margin 2027-03,DAU 2027-12,Cumul Margin 2027-12
Scenario,,,,
plan a.1 - 2026 ua250K + retention d30,42420,1.162266e+07,20052,1.475046e+07
plan a.1 - 2027q2 ua250k,54552,1.079348e+07,24560,1.442328e+07
plan a.1 - 2026 ua350K,42754,1.109881e+07,19912,1.422503e+07
plan a.1 - 2026 ua250K,39676,1.112374e+07,19198,1.409354e+07
plan a.1 - 2027 ua350K,63258,1.060906e+07,55735,1.408566e+07
plan a.1 - 2027 ua250K,54552,1.079348e+07,45327,1.406285e+07
plan a.1 - 2026q3 ua350k,34654,1.117273e+07,17952,1.387359e+07
"plan a - no ua, no uplifts",30483,1.111046e+07,16668,1.358021e+07


In [14]:
export_all_tables(actuals=actuals)

  saved: plan_a.1_-_2026_ua250K_pl_table.csv
  saved: plan_a.1_-_2026_ua250K_+_retention_d30_pl_table.csv
  saved: plan_a.1_-_2026_ua350K_pl_table.csv
  saved: plan_a.1_-_2026q3_ua350k_pl_table.csv
  saved: plan_a.1_-_2027_ua250K_pl_table.csv
  saved: plan_a.1_-_2027_ua350K_pl_table.csv
  saved: plan_a.1_-_2027q2_ua250k_pl_table.csv
  saved: plan_a_-_no_ua,_no_uplifts_pl_table.csv
  saved: comparison_table.csv

All exports written to: /Users/ivanaguilar/Desktop/DataStuff/gitrepos/testrepo/simulator/exports


PosixPath('/Users/ivanaguilar/Desktop/DataStuff/gitrepos/testrepo/simulator/exports')